In [8]:
import os
import pandas as pd
import numpy as np
import torch
import pickle
from torch.utils.data import Dataset


class InformerScenarioDataset(Dataset):

    bool_dtypes = [
        "RCP_pump", "HX", "HPI", "LPI", "CNMT_Spray", "MDAFW",
        "Charging_pump", "SAMG_1", "SAMG_2", "SAMG_3"
    ]

    def __init__(self, filepath_arr, seq_len=24, pred_len=1, log_dir="logs", cache_dir="cache"):
        self.filepath_arr = filepath_arr
        self.seq_len = seq_len
        self.pred_len = pred_len
        self.log_dir = log_dir
        self.cache_dir = cache_dir

        os.makedirs(log_dir, exist_ok=True)
        os.makedirs(cache_dir, exist_ok=True)

        self.all_data = None
        self.read_files()
        self.convert_data_types()
        self.series_index = self.arrange_indexes()

    def read_files(self):
        self.all_data = pd.read_csv(self.filepath_arr[0])
        assert self.all_data.isnull().sum().sum() == 0, "There should be no null values"
        duplicate_df = self.check_duplicates()
        duplicate_df.to_csv(f"{self.log_dir}/duplicates.csv")
        self.all_data = self.all_data.drop_duplicates().reset_index(drop=True)

    def check_duplicates(self):
        return self.all_data[self.all_data.duplicated(keep=False)]

    def convert_data_types(self):
        for col in self.bool_dtypes:
            self.all_data[col] = self.all_data[col].map(lambda x: 0 if x == 0.2 else 1)

    def get_cache_path(self):
        filename = os.path.basename(self.filepath_arr[0])
        cache_filename = filename + f"_s{self.seq_len}_p{self.pred_len}.cache"
        return os.path.join(self.cache_dir, cache_filename)

    def arrange_indexes(self):
        cache_path = self.get_cache_path()
        if os.path.exists(cache_path):
            with open(cache_path, "rb") as f:
                print(f"Loading cached indexes from {cache_path}")
                return pickle.load(f)

        print("Generating valid indexes...")
        valid_indexes = []
        df = self.all_data
        total_len = len(df)

        for i in range(total_len - self.seq_len - self.pred_len + 1):
            window = df.iloc[i:i + self.seq_len + self.pred_len]
            if window['scenario_number'].nunique() == 1:
                valid_indexes.append(i)

        with open(cache_path, "wb") as f:
            pickle.dump(valid_indexes, f)
            print(f"Cached indexes saved to {cache_path}")

        return valid_indexes

    def __getitem__(self, index):
        start_idx = self.series_index[index]
        end_idx = start_idx + self.seq_len
        label_idx = end_idx
        label_end_idx = label_idx + self.pred_len

        # Core values
        x = self.all_data.iloc[start_idx:end_idx].drop(columns=["scenario_number", "TIME"]).values
        y = self.all_data.iloc[label_idx:label_end_idx].drop(columns=["scenario_number","TIME"]).values

        # Time features (simple position encoding)
        past_time = np.linspace(0, 1, self.seq_len).reshape(-1, 1)
        future_time = np.linspace(1.01, 1.01 + self.pred_len / self.seq_len, self.pred_len).reshape(-1, 1)

        return {
            "past_values": torch.tensor(x, dtype=torch.float32),                          # [seq_len, num_features]
            "future_values": torch.tensor(y, dtype=torch.float32),                        # [pred_len, num_features]
            "past_time_features": torch.tensor(past_time, dtype=torch.float32),           # [seq_len, 1]
            "future_time_features": torch.tensor(future_time, dtype=torch.float32),       # [pred_len, 1]
            "past_observed_mask": torch.ones_like(torch.tensor(x, dtype=torch.float32)),  # Assume no missing
            "static_categorical_features": torch.tensor([0]),                              # Placeholder
            "static_real_features": torch.tensor([0.0])                                    # Placeholder
        }

    def get_debug_item(self, index):
        """
        Return the raw window slices with TIME and optionally scenario_number for debugging.
        Includes encoder input, decoder input range, and target.
        """
        start_idx = self.series_index[index]
        end_idx = start_idx + self.seq_len
        label_idx = end_idx
        label_end_idx = label_idx + self.pred_len

        # Extract dataframes with all columns (including TIME)
        enc_window = self.all_data.iloc[start_idx:end_idx].copy()
        dec_window = self.all_data.iloc[end_idx - 1:end_idx + self.pred_len].copy()
        target_window = self.all_data.iloc[label_idx:label_end_idx].copy()

        # Optional: add feature for position index within window (for readability)
        enc_window['__step__'] = range(-self.seq_len, 0)
        dec_window['__step__'] = range(-1, self.pred_len)
        target_window['__step__'] = range(0, self.pred_len)

        return {
            'enc_df': enc_window,       # The encoder input window [t-k:t)
            'dec_df': dec_window,       # The decoder input window [t-1:t+pred_len)
            'target_df': target_window  # The true target values [t:t+pred_len)
        }

    def __len__(self):
        return len(self.series_index)

In [9]:
data_path = "combined_sorted_data_margin_normalization.csv"
dataset = InformerScenarioDataset([data_path], seq_len=20, pred_len=1)


Loading cached indexes from cache/combined_sorted_data_margin_normalization.csv_s20_p1.cache


In [10]:
import torch
from torch.utils.data import DataLoader

In [11]:
# Simple DataLoader (no shuffle, batch size 1)
dataloader = DataLoader(dataset, batch_size=1, shuffle=False)

# Get feature dimension from one batch
sample = next(iter(dataloader))


In [13]:
from transformers import TimeSeriesTransformerConfig, TimeSeriesTransformerForPrediction


In [26]:
sample = dataset[0]  # Assuming your dataset object is named `dataset`
for k, v in sample.items():
    print(k, v.shape)

past_values torch.Size([20, 17])
future_values torch.Size([1, 17])
past_time_features torch.Size([20, 1])
future_time_features torch.Size([1, 1])
past_observed_mask torch.Size([20, 17])
static_categorical_features torch.Size([1])
static_real_features torch.Size([1])


In [27]:
# Constants
seq_len = 20
d = 17
pred_len = 1

In [28]:
batch = {k: v.unsqueeze(0) for k, v in sample.items()}  # Add batch dim
for k, v in batch.items():
    print(k, v.shape)

past_values torch.Size([1, 20, 17])
future_values torch.Size([1, 1, 17])
past_time_features torch.Size([1, 20, 1])
future_time_features torch.Size([1, 1, 1])
past_observed_mask torch.Size([1, 20, 17])
static_categorical_features torch.Size([1, 1])
static_real_features torch.Size([1, 1])


In [47]:
config = TimeSeriesTransformerConfig(
    prediction_length=1,
    context_length=20,
    input_size=17,
    num_time_features=1,
    num_static_categorical_features=1,
    num_static_real_features=1,
    lags_sequence= [0,1,2],  # from t-1 to t-20
    scaling=None,  # or try scaling=None to isolate error
    d_model=64,
    encoder_layers=2,
    decoder_layers=2,
)

In [51]:
assert sample['past_values'].shape[0] == config.context_length  # 20
assert sample['past_values'].shape[1] == config.input_size        # 17

In [49]:
model = TimeSeriesTransformerForPrediction(config)

In [50]:
model.train()
output = model(**batch)
print("Training loss:", output.loss.item())

RuntimeError: index_select(): self indexing axis dim should be positive

In [54]:

config = TimeSeriesTransformerConfig(
    prediction_length=24,
    # context length:
    context_length=48,
    # lags coming from helper given the freq:
    lags_sequence=[1, 2, 3, 4, 5, 6, 7, 11, 12, 13, 23, 24, 25, 35, 36, 37],
    # we'll add 2 time features ("month of year" and "age", see further):
    num_time_features=2,
    # we have a single static categorical feature, namely time series ID:
    num_static_categorical_features=1,
    # it has 366 possible values:
    cardinality=[366],
    # the model will learn an embedding of size 2 for each of the 366 possible values:
    embedding_dimension=[2],
    
    # transformer params:
    encoder_layers=4,
    decoder_layers=4,
    d_model=32,
)
model = TimeSeriesTransformerForPrediction(config)

In [55]:
batch = {
    'past_time_features': torch.randn(256, 85, 2).float(),  # shape [256, 85, 2], FloatTensor
    'past_values': torch.randn(256, 85).float(),              # shape [256, 85], FloatTensor
    'past_observed_mask': torch.randn(256, 85).float(),       # shape [256, 85], FloatTensor
    'future_time_features': torch.randn(256, 24, 2).float(),  # shape [256, 24, 2], FloatTensor
    'static_categorical_features': torch.randint(0, 10, (256, 1)).long(),  # shape [256, 1], LongTensor
    'future_values': torch.randn(256, 24).float(),            # shape [256, 24], FloatTensor
    'future_observed_mask': torch.randn(256, 24).float()      # shape [256, 24], FloatTensor
}

In [56]:
# perform forward pass
outputs = model(
    past_values=batch["past_values"],
    past_time_features=batch["past_time_features"],
    past_observed_mask=batch["past_observed_mask"],
    static_categorical_features=batch["static_categorical_features"]
    if config.num_static_categorical_features > 0
    else None,
    static_real_features=batch["static_real_features"]
    if config.num_static_real_features > 0
    else None,
    future_values=batch["future_values"],
    future_time_features=batch["future_time_features"],
    future_observed_mask=batch["future_observed_mask"],
    output_hidden_states=True,
)

In [57]:
print("Loss:", outputs.loss.item())

Loss: -154.15171813964844


In [61]:
print("hello")

hello


In [62]:
len(config.lags_sequence)

16